# 基于一阶马尔可夫链与 FPMC 的 MovieLens-1M 下一项电影推荐实验

本 Notebook 基于当前项目代码组织一个可顺序运行的实验，比较五种 next-item recommendation 方法：

| Model | Score | 说明 |
| --- | --- | --- |
| Popularity | `score(j)=count(j)` | 非序列热门 baseline |
| Markov Chain | `score(j)=P(j | i_t)` | 显式一阶马尔可夫转移矩阵 |
| MF-only | `<p_u, q_j>` | 只建模用户长期偏好 |
| FMC-only | `<a_i, b_j>` | 因子分解的一阶 Markov 转移 |
| FPMC | `<p_u,q_j> + <a_i,b_j>` | 用户长期偏好 + 短期 Markov 转移 |

普通 Markov Chain 显式统计 `item_i -> item_j` 的转移概率。FMC 用低维向量点积近似 `item_i -> item_j` 的转移分数。FPMC 进一步加入 user-item matrix factorization，使推荐同时考虑用户长期偏好和短期状态转移。

In [1]:
import importlib.util
import json
import math
import os
import random
import sys
import urllib.request
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

required_packages = ["numpy", "torch", "tqdm"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]
if missing_packages:
    raise ImportError(
        "Missing packages: " + ", ".join(missing_packages) + ".\n"
        "Install them in this notebook kernel, for example:\n"
        "%pip install torch numpy tqdm"
    )

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "models.py").exists() and (PROJECT_ROOT / "Factorized_Personalized_Markov_Chains" / "models.py").exists():
    PROJECT_ROOT = PROJECT_ROOT / "Factorized_Personalized_Markov_Chains"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dataset import BPR_Dataset
from metrics import BPR_Loss
from models import FPMC
from preprocessing import create_df, create_user_history, create_user_noclick, reset_df, train_val_test_split

CONFIG = {
    "seed": 42,
    "min_user_len": 5,
    "num_neg_eval": 100,
    "batch_size": 4096,
    "epochs": 100,
    "lr": 1e-3,
    "k_ui": 64,
    "k_il": 64,
    "weight_decay": 1e-6,
    "num_workers": 0,
    "max_train_samples": None,
    "eval_every": 1,
    "select_metric": "NDCG@10",
    "early_stop_patience": 3,
    "early_stop_min_delta": 1e-5,
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
print(torch.__version__)

device = cuda
2.5.1+cu124


## 下载 / 定位 MovieLens-1M

数据文件应位于 `ml-1m/ratings.dat`。如果不存在，将自动从 GroupLens 下载并解压。

In [2]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "ml-1m"
RATINGS_PATH = DATA_DIR / "ratings.dat"
ZIP_PATH = PROJECT_ROOT / "ml-1m.zip"
DATA_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

if not RATINGS_PATH.exists():
    print("ratings.dat not found, downloading MovieLens-1M...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(PROJECT_ROOT)
else:
    print(f"Found {RATINGS_PATH}")

assert RATINGS_PATH.exists(), "MovieLens-1M ratings.dat was not found or downloaded."

Found /root/autodl-tmp/Factorized_Personalized_Markov_Chains/ml-1m/ratings.dat


## 读取数据与 user/item 重编码

复用项目中的 `create_df` 和 `reset_df`。所有评分记录都作为交互行为，不按 rating 过滤。

In [3]:
ratings = create_df("ml-1m/ratings.dat")
encoder = reset_df()
ratings = encoder.fit_transform(ratings)

n_users = ratings["user_id"].nunique()
n_items = ratings["item_id"].nunique()
print(f"users={n_users:,}, items={n_items:,}, interactions={len(ratings):,}")
ratings.head()

========== Creating DataFrame ==========
{'user_id': 6040, 'item_id': 3706, 'rating': 5, 'timestamp': 458455}
(1000209, 4)
========== Initialize Reset DataFrame Object ==========
========== Resetting user ids and item ids in DataFrame ==========
users=6,040, items=3,706, interactions=1,000,209


[{'user_id': 6039, 'item_id': 802, 'rating': 4, 'timestamp': 956703932},
 {'user_id': 6039, 'item_id': 579, 'rating': 5, 'timestamp': 956703954},
 {'user_id': 6039, 'item_id': 2191, 'rating': 4, 'timestamp': 956703954},
 {'user_id': 6039, 'item_id': 1781, 'rating': 4, 'timestamp': 956703977},
 {'user_id': 6039, 'item_id': 1839, 'rating': 5, 'timestamp': 956703977}]

## 时间序列划分

使用 leave-one-out sequential split。每个用户按时间得到 `i1 -> i2 -> ... -> in`，训练集使用除最后两个转移外的全部转移，验证集使用倒数第二个转移，测试集使用最后一个转移。样本形式为 `(u, prev_item, next_item)`。

In [4]:
user_history_all = create_user_history(ratings)
user_history = {u: hist for u, hist in user_history_all.items() if len(hist) >= CONFIG["min_user_len"]}
train_samples, val_samples, test_samples = train_val_test_split(user_history)

print(f"eligible users={len(user_history):,}")
print(f"train={len(train_samples):,}, val={len(val_samples):,}, test={len(test_samples):,}")
train_samples[:3], val_samples[:3], test_samples[:3]

========== Creating User Histories ==========


100%|██████████| 1000209/1000209 [00:00<00:00, 2013802.04it/s]


========== Splitting User Histories into Train, Validation, and Test Splits ==========


100%|██████████| 6040/6040 [00:00<00:00, 35986.24it/s]

eligible users=6,040
train=982,089, val=6,040, test=6,040


([(6039, 802, 579), (6039, 579, 2191), (6039, 2191, 1781)],
 [(6039, 1618, 155), (6038, 861, 1114), (6037, 1316, 2495)],
 [(6039, 155, 1131), (6038, 1114, 1162), (6037, 2495, 1094)])

## 构造 user_seen 与固定评估候选集

sampled ranking 对每个验证/测试样本使用同一套固定候选集：`[真实 next item] + 100 个负样本`。后续所有模型都在完全相同的候选集合上评估。

In [5]:
user_seen = {u: set(hist) for u, hist in user_history.items()}
user_noclick = create_user_noclick(user_history, ratings, n_items)

def build_eval_candidates(samples, user_noclick, num_neg, seed):
    rng = np.random.default_rng(seed)
    candidate_rows = []
    for u, prev_i, true_i in samples:
        negatives, probabilities = user_noclick[u]
        replace = len(negatives) < num_neg
        sampled = rng.choice(negatives, size=num_neg, replace=replace, p=probabilities).tolist()
        candidate_rows.append([true_i] + sampled)
    return np.asarray(candidate_rows, dtype=np.int64)

val_candidates = build_eval_candidates(val_samples, user_noclick, CONFIG["num_neg_eval"], CONFIG["seed"] + 1)
test_candidates = build_eval_candidates(test_samples, user_noclick, CONFIG["num_neg_eval"], CONFIG["seed"] + 2)
print(val_candidates.shape, test_candidates.shape)

========== Creating User 'no-click' history ==========


100%|██████████| 6040/6040 [00:01<00:00, 3691.26it/s]


(6040, 101) (6040, 101)


## Popularity baseline

Popularity 不使用用户或上一项，只按训练集中 item 出现次数打分：`score(j)=count(j)`。

In [6]:
item_popularity = np.zeros(n_items, dtype=np.float32)
for _, _, next_i in train_samples:
    item_popularity[next_i] += 1.0

def score_popularity(samples, candidates):
    return item_popularity[candidates]

print("Top popular encoded item ids:", np.argsort(-item_popularity)[:10].tolist())

Top popular encoded item ids: [2651, 1106, 253, 1120, 575, 1848, 2374, 466, 1449, 593]


## Markov Chain baseline

显式统计训练集中 `prev_item -> next_item` 的一阶转移概率：`score(j)=P(j | i_t)`。如果某个上一项没有观测到转移，则退回到 popularity 分数。

In [7]:
transition_counts = defaultdict(Counter)
transition_totals = Counter()
for _, prev_i, next_i in train_samples:
    transition_counts[prev_i][next_i] += 1
    transition_totals[prev_i] += 1

def score_markov(samples, candidates):
    scores = np.zeros_like(candidates, dtype=np.float32)
    for row_idx, (_, prev_i, _) in enumerate(samples):
        total = transition_totals[prev_i]
        if total == 0:
            scores[row_idx] = item_popularity[candidates[row_idx]]
            continue
        row_counts = transition_counts[prev_i]
        scores[row_idx] = [row_counts[int(item)] / total for item in candidates[row_idx]]
    return scores

print(f"observed previous items with transitions={len(transition_counts):,}")

observed previous items with transitions=3,704


## FPMC 模型定义

本项目的 `models.FPMC` 已支持三种模式：

- `mode="mf"`: `s_MF(u,j)=<p_u,q_j>`
- `mode="fmc"`: `s_FMC(i,j)=<a_i,b_j>`
- `mode="full"`: `s_FPMC(u,i,j)=s_MF(u,j)+s_FMC(i,j)`

In [8]:
for mode in ["mf", "fmc", "full"]:
    probe_model = FPMC(n_users, n_items, CONFIG["k_ui"], CONFIG["k_il"], mode=mode)
    probe_out = probe_model(torch.tensor([0]), torch.tensor([0]), torch.tensor([0]))
    print(mode, tuple(probe_out.shape))
del probe_model, probe_out

========== Creating FPMC Model ==========
mf (1,)
========== Creating FPMC Model ==========
fmc (1,)
========== Creating FPMC Model ==========
full (1,)


## 训练函数与评估函数

训练使用项目中的 `BPR_Dataset` 和 `BPR_Loss`。BPR pairwise loss 为：

```python
F.softplus(-(pos_scores - neg_scores)).mean()
```

评估指标包括 `Hit@5`、`Hit@10`、`MRR@10`、`NDCG@10`。候选集中真实 item 固定在第 0 列。

In [9]:
def bpr_collate_fn(batch):
    uid, prev_i, pos_i, neg_i = zip(*batch)
    return (
        torch.tensor(uid, dtype=torch.long),
        torch.tensor(prev_i, dtype=torch.long),
        torch.tensor(pos_i, dtype=torch.long),
        torch.tensor(neg_i, dtype=torch.long),
    )

def make_train_loader(samples):
    dataset = BPR_Dataset(samples, n_users, n_items)
    if CONFIG["max_train_samples"] is not None and len(dataset) > CONFIG["max_train_samples"]:
        rng = np.random.default_rng(CONFIG["seed"])
        indices = rng.choice(len(dataset), size=CONFIG["max_train_samples"], replace=False)
        dataset = Subset(dataset, indices.tolist())
        print(f"Using {len(dataset):,} sampled training transitions.")
    else:
        print(f"Using full training transitions: {len(dataset):,}.")

    generator = torch.Generator()
    generator.manual_seed(CONFIG["seed"])
    return DataLoader(
        dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=CONFIG["num_workers"],
        collate_fn=bpr_collate_fn,
        generator=generator,
    )

def score_torch_model(model, samples, candidates, batch_size=4096):
    model.eval()
    all_scores = []
    with torch.no_grad():
        for start in range(0, len(samples), batch_size):
            end = min(start + batch_size, len(samples))
            batch_samples = samples[start:end]
            batch_candidates = candidates[start:end]
            width = batch_candidates.shape[1]

            uid = torch.tensor([s[0] for s in batch_samples], dtype=torch.long, device=device).repeat_interleave(width)
            prev_i = torch.tensor([s[1] for s in batch_samples], dtype=torch.long, device=device).repeat_interleave(width)
            cand_i = torch.tensor(batch_candidates.reshape(-1), dtype=torch.long, device=device)

            scores = model(uid, prev_i, cand_i).reshape(end - start, width)
            all_scores.append(scores.cpu().numpy())
    return np.vstack(all_scores)

def ranking_metrics(scores):
    order = np.argsort(-scores, axis=1)
    true_positions = np.argwhere(order == 0)[:, 1] + 1
    hit5 = np.mean(true_positions <= 5)
    hit10 = np.mean(true_positions <= 10)
    mrr10 = np.mean(np.where(true_positions <= 10, 1.0 / true_positions, 0.0))
    ndcg10 = np.mean(np.where(true_positions <= 10, 1.0 / np.log2(true_positions + 1), 0.0))
    return {"Hit@5": float(hit5), "Hit@10": float(hit10), "MRR@10": float(mrr10), "NDCG@10": float(ndcg10)}

def evaluate_scores(name, scores, extra=None):
    row = {"Model": name}
    row.update(ranking_metrics(scores))
    if extra:
        row.update(extra)
    return row

def train_torch_model(mode, display_name):
    set_seed(CONFIG["seed"])
    model = FPMC(n_users, n_items, CONFIG["k_ui"], CONFIG["k_il"], mode=mode).to(device)
    criterion = BPR_Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    train_loader = make_train_loader(train_samples)

    loss_history = []
    val_history = []
    best_epoch = 0
    best_score = -float("inf")
    best_state = None
    bad_epochs = 0
    select_metric = CONFIG["select_metric"]
    patience = CONFIG["early_stop_patience"]
    min_delta = CONFIG["early_stop_min_delta"]

    for epoch in range(1, CONFIG["epochs"] + 1):
        model.train()
        losses = []
        for uid, prev_i, pos_i, neg_i in tqdm(train_loader, desc=f"{mode} epoch {epoch}", leave=False):
            uid = uid.to(device)
            prev_i = prev_i.to(device)
            pos_i = pos_i.to(device)
            neg_i = neg_i.to(device)

            optimizer.zero_grad()
            pos_scores = model(uid, prev_i, pos_i)
            neg_scores = model(uid, prev_i, neg_i)
            loss = criterion(pos_scores, neg_scores)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        mean_loss = float(np.mean(losses))
        loss_history.append(mean_loss)

        if epoch % CONFIG["eval_every"] == 0 or epoch == CONFIG["epochs"]:
            val_metrics = ranking_metrics(score_torch_model(model, val_samples, val_candidates))
            val_history.append({"epoch": epoch, "loss": mean_loss, **val_metrics})
            current_score = val_metrics[select_metric]
            improved = current_score > best_score + min_delta
            if improved:
                best_score = current_score
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
            print(
                f"{display_name} epoch {epoch:02d}: "
                f"loss={mean_loss:.4f}, val {select_metric}={current_score:.4f}, "
                f"best_epoch={best_epoch}, no_improve={bad_epochs}/{patience}"
            )
            if bad_epochs >= patience:
                print(
                    f"Early stopping {display_name}: validation {select_metric} "
                    f"did not improve for {patience} consecutive evaluations."
                )
                break
        else:
            print(f"{display_name} epoch {epoch:02d}: loss={mean_loss:.4f}")

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    return model, loss_history, val_history, best_epoch, best_score

MODEL_ORDER = ["Popularity", "Markov Chain", "MF-only", "FMC-only", "FPMC"]
METRIC_COLUMNS = ["Hit@5", "Hit@10", "MRR@10", "NDCG@10"]
META_COLUMNS = ["BestEpoch", "ValNDCG@10"]

def ordered_results(rows):
    by_model = {row["Model"]: row for row in rows}
    return [by_model[name] for name in MODEL_ORDER if name in by_model]

def display_results(rows):
    rows = ordered_results(rows)
    header = "| Model | Hit@5 | Hit@10 | MRR@10 | NDCG@10 | Best epoch | Val NDCG@10 |\n| --- | ---: | ---: | ---: | ---: | ---: | ---: |"
    body = []
    for row in rows:
        best_epoch = row.get("BestEpoch", "-")
        val_ndcg = row.get("ValNDCG@10", None)
        val_text = "-" if val_ndcg is None else f"{val_ndcg:.4f}"
        body.append(
            f"| {row['Model']} | {row['Hit@5']:.4f} | {row['Hit@10']:.4f} | "
            f"{row['MRR@10']:.4f} | {row['NDCG@10']:.4f} | {best_epoch} | {val_text} |"
        )
    table = header + "\n" + "\n".join(body)
    try:
        from IPython.display import Markdown, display
        display(Markdown(table))
    except Exception:
        print(table)
    return rows

results = []
results.append(evaluate_scores("Popularity", score_popularity(test_samples, test_candidates)))
results.append(evaluate_scores("Markov Chain", score_markov(test_samples, test_candidates)))
display_results(results)

| Model | Hit@5 | Hit@10 | MRR@10 | NDCG@10 | Best epoch | Val NDCG@10 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Popularity | 0.0724 | 0.1356 | 0.0426 | 0.0639 | - | - |
| Markov Chain | 0.4146 | 0.4974 | 0.3025 | 0.3491 | - | - |

[{'Model': 'Popularity',
  'Hit@5': 0.07235099337748345,
  'Hit@10': 0.13559602649006622,
  'MRR@10': 0.04260479081257227,
  'NDCG@10': 0.06392988505269162},
 {'Model': 'Markov Chain',
  'Hit@5': 0.41456953642384103,
  'Hit@10': 0.49735099337748345,
  'MRR@10': 0.3025298276043309,
  'NDCG@10': 0.3490607838506044}]

## 训练 MF-only

In [10]:
mf_model, mf_loss, mf_val_history, mf_best_epoch, mf_best_val = train_torch_model("mf", "MF-only")
results.append(evaluate_scores(
    "MF-only",
    score_torch_model(mf_model, test_samples, test_candidates),
    {"BestEpoch": mf_best_epoch, "ValNDCG@10": mf_best_val},
))
display_results(results)

========== Creating FPMC Model ==========
========== Creating Model Loss Criterion (s-BPR) ==========
========== Creating PyTorch Dataset Object ==========
Using full training transitions: 982,089.


mf epoch 1:   0%|          | 0/240 [00:00<?, ?it/s]

MF-only epoch 01: loss=0.8902, val NDCG@10=0.0440, best_epoch=1, no_improve=0/3


mf epoch 2:   0%|          | 0/240 [00:00<?, ?it/s]

MF-only epoch 02: loss=0.8690, val NDCG@10=0.0442, best_epoch=2, no_improve=0/3


...

MF-only epoch 04: loss=0.8315, val NDCG@10=0.0441, best_epoch=2, no_improve=2/3


mf epoch 5:   0%|          | 0/240 [00:00<?, ?it/s]

MF-only epoch 05: loss=0.8155, val NDCG@10=0.0440, best_epoch=2, no_improve=3/3
Early stopping MF-only: validation NDCG@10 did not improve for 3 consecutive evaluations.


| Model | Hit@5 | Hit@10 | MRR@10 | NDCG@10 | Best epoch | Val NDCG@10 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Popularity | 0.0724 | 0.1356 | 0.0426 | 0.0639 | - | - |
| Markov Chain | 0.4146 | 0.4974 | 0.3025 | 0.3491 | - | - |
| MF-only | 0.0478 | 0.0972 | 0.0282 | 0.0439 | 2 | 0.0442 |

[{'Model': 'Popularity',
  'Hit@5': 0.07235099337748345,
  'Hit@10': 0.13559602649006622,
  'MRR@10': 0.04260479081257227,
  'NDCG@10': 0.06392988505269162},
 {'Model': 'Markov Chain',
  'Hit@5': 0.41456953642384103,
  'Hit@10': 0.49735099337748345,
  'MRR@10': 0.3025298276043309,
  'NDCG@10': 0.3490607838506044},
 {'Model': 'MF-only',
  'Hit@5': 0.0478476821192053,
  'Hit@10': 0.09718543046357615,
  'MRR@10': 0.028188925680647535,
  'NDCG@10': 0.043923459464553005,
  'BestEpoch': 2,
  'ValNDCG@10': 0.04419863756111363}]

## 训练 FMC-only

In [11]:
fmc_model, fmc_loss, fmc_val_history, fmc_best_epoch, fmc_best_val = train_torch_model("fmc", "FMC-only")
results.append(evaluate_scores(
    "FMC-only",
    score_torch_model(fmc_model, test_samples, test_candidates),
    {"BestEpoch": fmc_best_epoch, "ValNDCG@10": fmc_best_val},
))
display_results(results)

========== Creating FPMC Model ==========
========== Creating Model Loss Criterion (s-BPR) ==========
========== Creating PyTorch Dataset Object ==========
Using full training transitions: 982,089.


fmc epoch 1:   0%|          | 0/240 [00:00<?, ?it/s]

FMC-only epoch 01: loss=0.8848, val NDCG@10=0.0458, best_epoch=1, no_improve=0/3


fmc epoch 2:   0%|          | 0/240 [00:00<?, ?it/s]

FMC-only epoch 02: loss=0.8464, val NDCG@10=0.0493, best_epoch=2, no_improve=0/3


...

FMC-only epoch 98: loss=0.2092, val NDCG@10=0.3184, best_epoch=96, no_improve=2/3


fmc epoch 99:   0%|          | 0/240 [00:00<?, ?it/s]

FMC-only epoch 99: loss=0.2084, val NDCG@10=0.3189, best_epoch=96, no_improve=3/3
Early stopping FMC-only: validation NDCG@10 did not improve for 3 consecutive evaluations.


| Model | Hit@5 | Hit@10 | MRR@10 | NDCG@10 | Best epoch | Val NDCG@10 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Popularity | 0.0724 | 0.1356 | 0.0426 | 0.0639 | - | - |
| Markov Chain | 0.4146 | 0.4974 | 0.3025 | 0.3491 | - | - |
| MF-only | 0.0478 | 0.0972 | 0.0282 | 0.0439 | 2 | 0.0442 |
| FMC-only | 0.3725 | 0.4921 | 0.2416 | 0.3006 | 96 | 0.3190 |

[{'Model': 'Popularity',
  'Hit@5': 0.07235099337748345,
  'Hit@10': 0.13559602649006622,
  'MRR@10': 0.04260479081257227,
  'NDCG@10': 0.06392988505269162},
 {'Model': 'Markov Chain',
  'Hit@5': 0.41456953642384103,
  'Hit@10': 0.49735099337748345,
  'MRR@10': 0.3025298276043309,
  'NDCG@10': 0.3490607838506044},
 {'Model': 'MF-only',
  'Hit@5': 0.0478476821192053,
  'Hit@10': 0.09718543046357615,
  'MRR@10': 0.028188925680647535,
  'NDCG@10': 0.043923459464553005,
  'BestEpoch': 2,
  'ValNDCG@10': 0.04419863756111363},
 {'Model': 'FMC-only',
  'Hit@5': 0.37251655629139074,
  'Hit@10': 0.49205298013245036,
  'MRR@10': 0.24161029643645537,
  'NDCG@10': 0.3006462337586254,
  'BestEpoch': 96,
  'ValNDCG@10': 0.31896376899367607}]

## 训练 FPMC

In [12]:
fpmc_model, fpmc_loss, fpmc_val_history, fpmc_best_epoch, fpmc_best_val = train_torch_model("full", "FPMC")
results.append(evaluate_scores(
    "FPMC",
    score_torch_model(fpmc_model, test_samples, test_candidates),
    {"BestEpoch": fpmc_best_epoch, "ValNDCG@10": fpmc_best_val},
))
display_results(results)

========== Creating FPMC Model ==========
========== Creating Model Loss Criterion (s-BPR) ==========
========== Creating PyTorch Dataset Object ==========
Using full training transitions: 982,089.


full epoch 1:   0%|          | 0/240 [00:00<?, ?it/s]

FPMC epoch 01: loss=1.0443, val NDCG@10=0.0462, best_epoch=1, no_improve=0/3


full epoch 2:   0%|          | 0/240 [00:00<?, ?it/s]

FPMC epoch 02: loss=0.9931, val NDCG@10=0.0486, best_epoch=2, no_improve=0/3


...

FPMC epoch 83: loss=0.1905, val NDCG@10=0.3252, best_epoch=81, no_improve=2/3


full epoch 84:   0%|          | 0/240 [00:00<?, ?it/s]

FPMC epoch 84: loss=0.1902, val NDCG@10=0.3255, best_epoch=81, no_improve=3/3
Early stopping FPMC: validation NDCG@10 did not improve for 3 consecutive evaluations.


| Model | Hit@5 | Hit@10 | MRR@10 | NDCG@10 | Best epoch | Val NDCG@10 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Popularity | 0.0724 | 0.1356 | 0.0426 | 0.0639 | - | - |
| Markov Chain | 0.4146 | 0.4974 | 0.3025 | 0.3491 | - | - |
| MF-only | 0.0478 | 0.0972 | 0.0282 | 0.0439 | 2 | 0.0442 |
| FMC-only | 0.3725 | 0.4921 | 0.2416 | 0.3006 | 96 | 0.3190 |
| FPMC | 0.3825 | 0.5081 | 0.2484 | 0.3094 | 81 | 0.3255 |

[{'Model': 'Popularity',
  'Hit@5': 0.07235099337748345,
  'Hit@10': 0.13559602649006622,
  'MRR@10': 0.04260479081257227,
  'NDCG@10': 0.06392988505269162},
 {'Model': 'Markov Chain',
  'Hit@5': 0.41456953642384103,
  'Hit@10': 0.49735099337748345,
  'MRR@10': 0.3025298276043309,
  'NDCG@10': 0.3490607838506044},
 {'Model': 'MF-only',
  'Hit@5': 0.0478476821192053,
  'Hit@10': 0.09718543046357615,
  'MRR@10': 0.028188925680647535,
  'NDCG@10': 0.043923459464553005,
  'BestEpoch': 2,
  'ValNDCG@10': 0.04419863756111363},
 {'Model': 'FMC-only',
  'Hit@5': 0.37251655629139074,
  'Hit@10': 0.49205298013245036,
  'MRR@10': 0.24161029643645537,
  'NDCG@10': 0.3006462337586254,
  'BestEpoch': 96,
  'ValNDCG@10': 0.31896376899367607},
 {'Model': 'FPMC',
  'Hit@5': 0.3824503311258278,
  'Hit@10': 0.508112582781457,
  'MRR@10': 0.24839449963208243,
  'NDCG@10': 0.3094402113661458,
  'BestEpoch': 81,
  'ValNDCG@10': 0.3254624420531981}]

## 结果汇总表

最终结果保存到：

- `results/fpmc_markov_results.csv`
- `results/fpmc_markov_results.json`

In [13]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(exist_ok=True)

import csv

results_table = display_results(results)

with open(results_dir / "fpmc_markov_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["Model"] + METRIC_COLUMNS + META_COLUMNS)
    writer.writeheader()
    writer.writerows(results_table)

with open(results_dir / "fpmc_markov_results.json", "w") as f:
    json.dump({row["Model"]: {key: row.get(key) for key in METRIC_COLUMNS + META_COLUMNS} for row in results_table}, f, indent=2)

print(results_dir / "fpmc_markov_results.csv")
print(results_dir / "fpmc_markov_results.json")

with open(results_dir / "fpmc_markov_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
print(results_dir / "fpmc_markov_config.json")

training_history = {
    "MF-only": {"loss": mf_loss, "validation": mf_val_history},
    "FMC-only": {"loss": fmc_loss, "validation": fmc_val_history},
    "FPMC": {"loss": fpmc_loss, "validation": fpmc_val_history},
}
with open(results_dir / "fpmc_markov_training_history.json", "w") as f:
    json.dump(training_history, f, indent=2)
print(results_dir / "fpmc_markov_training_history.json")

| Model | Hit@5 | Hit@10 | MRR@10 | NDCG@10 | Best epoch | Val NDCG@10 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Popularity | 0.0724 | 0.1356 | 0.0426 | 0.0639 | - | - |
| Markov Chain | 0.4146 | 0.4974 | 0.3025 | 0.3491 | - | - |
| MF-only | 0.0478 | 0.0972 | 0.0282 | 0.0439 | 2 | 0.0442 |
| FMC-only | 0.3725 | 0.4921 | 0.2416 | 0.3006 | 96 | 0.3190 |
| FPMC | 0.3825 | 0.5081 | 0.2484 | 0.3094 | 81 | 0.3255 |

/root/autodl-tmp/Factorized_Personalized_Markov_Chains/results/fpmc_markov_results.csv
/root/autodl-tmp/Factorized_Personalized_Markov_Chains/results/fpmc_markov_results.json
/root/autodl-tmp/Factorized_Personalized_Markov_Chains/results/fpmc_markov_config.json
/root/autodl-tmp/Factorized_Personalized_Markov_Chains/results/fpmc_markov_training_history.json


## 可视化

In [14]:
from html import escape

TABLE_STYLE = "border-collapse:collapse;font-family:sans-serif;font-size:13px;min-width:680px"
TH_STYLE = "padding:8px 12px;text-align:right;border-bottom:1px solid #cbd5e1;background:#f8fafc"
TD_STYLE = "padding:7px 12px;text-align:right;border-bottom:1px solid #e2e8f0"
MODEL_STYLE = "padding:7px 12px;text-align:left;border-bottom:1px solid #e2e8f0;font-weight:600"
BEST_STYLE = TD_STYLE + ";background:#dcfce7;font-weight:700;color:#166534"


def html_metric_comparison_table(rows, metrics):
    best_values = {metric: max(float(row[metric]) for row in rows) for metric in metrics}
    html = ["<h3>Formal test comparison</h3>"]
    html.append(f"<table style='{TABLE_STYLE}'>")
    html.append("<tr><th style='padding:8px 12px;text-align:left;border-bottom:1px solid #cbd5e1;background:#f8fafc'>Model</th>")
    for metric in metrics:
        html.append(f"<th style='{TH_STYLE}'>{escape(metric)}</th>")
    html.append(f"<th style='{TH_STYLE}'>Best epoch</th>")
    html.append(f"<th style='{TH_STYLE}'>Val NDCG@10</th>")
    html.append("</tr>")

    for row in rows:
        html.append(f"<tr><td style='{MODEL_STYLE}'>{escape(row['Model'])}</td>")
        for metric in metrics:
            value = float(row[metric])
            style = BEST_STYLE if value == best_values[metric] else TD_STYLE
            marker = " ✓" if value == best_values[metric] else ""
            html.append(f"<td style='{style}'>{value:.4f}{marker}</td>")
        best_epoch = row.get("BestEpoch", "-")
        val_ndcg = row.get("ValNDCG@10", None)
        val_text = "-" if val_ndcg is None else f"{float(val_ndcg):.4f}"
        html.append(f"<td style='{TD_STYLE}'>{best_epoch}</td>")
        html.append(f"<td style='{TD_STYLE}'>{val_text}</td>")
        html.append("</tr>")
    html.append("</table>")
    return "".join(html)

def html_loss_summary_table(loss_histories):
    html = ["<h3>Training loss summary</h3>"]
    html.append(f"<table style='{TABLE_STYLE}'>")
    html.append(
        "<tr>"
        "<th style='padding:8px 12px;text-align:left;border-bottom:1px solid #cbd5e1;background:#f8fafc'>Model</th>"
        f"<th style='{TH_STYLE}'>Epochs</th>"
        f"<th style='{TH_STYLE}'>Initial loss</th>"
        f"<th style='{TH_STYLE}'>Final loss</th>"
        f"<th style='{TH_STYLE}'>Best loss</th>"
        "</tr>"
    )
    for name, values in loss_histories.items():
        values = [float(v) for v in values]
        html.append(
            "<tr>"
            f"<td style='{MODEL_STYLE}'>{escape(name)}</td>"
            f"<td style='{TD_STYLE}'>{len(values)}</td>"
            f"<td style='{TD_STYLE}'>{values[0]:.4f}</td>"
            f"<td style='{TD_STYLE}'>{values[-1]:.4f}</td>"
            f"<td style='{TD_STYLE}'>{min(values):.4f}</td>"
            "</tr>"
        )
    html.append("</table>")
    return "".join(html)

try:
    from IPython.display import HTML, display
    display(HTML(html_metric_comparison_table(results_table, METRIC_COLUMNS)))
    display(HTML(html_loss_summary_table({"MF-only": mf_loss, "FMC-only": fmc_loss, "FPMC": fpmc_loss})))
except Exception:
    display_results(results_table)
    print({"MF-only": mf_loss, "FMC-only": fmc_loss, "FPMC": fpmc_loss})

Model,Hit@5,Hit@10,MRR@10,NDCG@10,Best epoch,Val NDCG@10
Popularity,0.0724,0.1356,0.0426,0.0639,-,-
Markov Chain,0.4146 ✓,0.4974,0.3025 ✓,0.3491 ✓,-,-
MF-only,0.0478,0.0972,0.0282,0.0439,2,0.0442
FMC-only,0.3725,0.4921,0.2416,0.3006,96,0.3190
FPMC,0.3825,0.5081 ✓,0.2484,0.3094,81,0.3255


Model,Epochs,Initial loss,Final loss,Best loss
MF-only,5,0.8902,0.8155,0.8155
FMC-only,99,0.8848,0.2084,0.2084
FPMC,84,1.0443,0.1902,0.1902


## 实验结论

本实验用 sampled ranking 比较了非序列 popularity、显式一阶 Markov Chain，以及三个基于向量因子分解的 PyTorch 模型。Popularity 反映全局热门偏置；显式 Markov Chain 只使用上一部电影到下一部电影的统计转移；MF-only 只使用用户长期偏好；FMC-only 用低维向量表示短期转移；FPMC 将长期偏好与短期转移相加。

当前配置用于正式对比：使用完整训练转移，以 50 个 epoch 作为训练上限，并按验证集 `CONFIG["select_metric"]` 选择 PyTorch 模型的最佳 epoch 后再报告测试集结果。若验证指标连续 `CONFIG["early_stop_patience"]` 次评估没有提升，则提前停止训练。MovieLens 评分序列并不一定具有很强的短期转移模式，因此 FPMC 不显著优于 FMC-only 或 MF-only 并不必然表示实现失败，需要结合训练轮数、负采样规模和候选评估协议一起判断。